# E-commerce Sales Data Analysis with Python and Pandas

This Colab-ready notebook loads the e-commerce sales CSV and analyzes sales by category, city, product, and payment method. It demonstrates data selection, filtering, sorting, `groupby()`, and aggregation.

**Before running:** download the CSV from the LMS **Study Material** tab, then run the upload cell below and choose that file.

In [ ]:
# Import required libraries
import io
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
sns.set_theme(style='whitegrid', palette='deep')

In [ ]:
# Upload the CSV in Google Colab.
# If using Jupyter locally, replace this cell with: file_name = 'your_file.csv'
try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No file was uploaded.')
    file_name, file_bytes = next(iter(uploaded.items()))
    df = pd.read_csv(io.BytesIO(file_bytes))
except ImportError:
    csv_files = list(Path('.').glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('Place the LMS CSV in this folder, then run the cell again.')
    file_name = csv_files[0]
    df = pd.read_csv(file_name)

print(f'Loaded: {file_name}')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Inspect the dataset and standardize column names for reliable analysis
print('Original columns:')
print(df.columns.tolist())
print('\nMissing values per column:')
display(df.isna().sum().to_frame('Missing values'))

def clean_name(column):
    return re.sub(r'[^a-z0-9]+', '_', str(column).strip().lower()).strip('_')

df.columns = [clean_name(col) for col in df.columns]

# Match frequent column-name variations used in e-commerce datasets.
aliases = {
    'sales': ['sales', 'total', 'total_sales', 'revenue', 'amount', 'order_total'],
    'category': ['category', 'product_line', 'product_category'],
    'product': ['product', 'product_name', 'item', 'item_name'],
    'city': ['city', 'location'],
    'payment_method': ['payment_method', 'payment', 'payment_type', 'paymentmode'],
    'quantity': ['quantity', 'qty', 'units_sold', 'units'],
    'order_id': ['order_id', 'invoice_id', 'invoice', 'transaction_id', 'id']
}

def first_present(candidates):
    return next((name for name in candidates if name in df.columns), None)

col = {key: first_present(names) for key, names in aliases.items()}

# Many supermarket-sales datasets store products as product lines.
if col['product'] is None and col['category'] is not None:
    col['product'] = col['category']

# Derive sales if the source has unit price and quantity but no total-sales field.
if col['sales'] is None:
    unit_price = first_present(['unit_price', 'price', 'unitprice'])
    if unit_price and col['quantity']:
        df['sales'] = pd.to_numeric(df[unit_price], errors='coerce') * pd.to_numeric(df[col['quantity']], errors='coerce')
        col['sales'] = 'sales'

required = ['sales', 'category', 'city', 'payment_method', 'quantity', 'order_id']
missing = [name for name in required if col[name] is None]
if missing:
    raise KeyError(f'Could not identify these required fields: {missing}. Review the aliases dictionary above for your CSV.')

# Convert the measure columns to numeric and remove records that cannot be analyzed.
df[col['sales']] = pd.to_numeric(df[col['sales']], errors='coerce')
df[col['quantity']] = pd.to_numeric(df[col['quantity']], errors='coerce')
analysis_df = df.dropna(subset=[col['sales'], col['quantity']]).copy()
print('Columns used for analysis:', col)
print(f'Usable records: {len(analysis_df):,} of {len(df):,}')

In [ ]:
# Overall statistics using selection and aggregation functions
sales_col, qty_col, order_col = col['sales'], col['quantity'], col['order_id']
overall_stats = pd.Series({
    'Total sales': analysis_df[sales_col].sum(),
    'Average sale per record': analysis_df[sales_col].mean(),
    'Highest sale': analysis_df[sales_col].max(),
    'Lowest sale': analysis_df[sales_col].min(),
    'Total quantity sold': analysis_df[qty_col].sum(),
    'Number of orders': analysis_df[order_col].nunique()
}, name='Value')
display(overall_stats.to_frame())

# Select useful fields from the first five transactions
selected_columns = [c for c in [order_col, col['city'], col['category'], col['product'], sales_col, qty_col, col['payment_method']] if c]
analysis_df[selected_columns].head()

In [ ]:
# Grouped analysis: categories, cities, products, and payment methods
def sales_summary(group_column):
    return (analysis_df.groupby(group_column)
            .agg(total_sales=(sales_col, 'sum'),
                 average_sales=(sales_col, 'mean'),
                 highest_sale=(sales_col, 'max'),
                 lowest_sale=(sales_col, 'min'),
                 total_quantity=(qty_col, 'sum'),
                 number_of_orders=(order_col, 'nunique'))
            .sort_values('total_sales', ascending=False))

category_summary = sales_summary(col['category'])
city_summary = sales_summary(col['city'])
payment_summary = sales_summary(col['payment_method'])
product_summary = sales_summary(col['product'])

print('Sales by category')
display(category_summary)
print('Sales by city')
display(city_summary)
print('Sales by payment method')
display(payment_summary)
print('Top 10 products/categories by total sales')
display(product_summary.head(10))

In [ ]:
# Filtering and sorting examples
# 1. Orders whose sales are in the top 25%
high_value_cutoff = analysis_df[sales_col].quantile(0.75)
high_value_orders = (analysis_df.loc[analysis_df[sales_col] >= high_value_cutoff, selected_columns]
                     .sort_values(sales_col, ascending=False))
print(f'High-value orders (sales >= {high_value_cutoff:,.2f})')
display(high_value_orders.head(10))

# 2. A filter for a particular city and payment method. Change the two values if needed.
target_city = analysis_df[col['city']].dropna().iloc[0]
target_payment = analysis_df[col['payment_method']].dropna().iloc[0]
city_payment_orders = (analysis_df.loc[
    (analysis_df[col['city']] == target_city) &
    (analysis_df[col['payment_method']] == target_payment), selected_columns]
    .sort_values(sales_col, ascending=False))
print(f'Orders in {target_city} paid using {target_payment}')
display(city_payment_orders.head(10))

# 3. Top-performing category/product and city using sorted group summaries
display(category_summary.head(3))
display(city_summary.head(3))

In [ ]:
# Cross-tabulation: total sales for each city-category combination
city_category_sales = pd.pivot_table(
    analysis_df, index=col['city'], columns=col['category'], values=sales_col,
    aggfunc='sum', fill_value=0
)
display(city_category_sales)

plt.figure(figsize=(12, 5))
sns.barplot(data=category_summary.reset_index(), x='total_sales', y=col['category'], color='#4C78A8')
plt.title('Total Sales by Category')
plt.xlabel('Total sales')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
# Automatically generated observations based on the actual dataset
top_category, top_category_row = next(iter(category_summary.iterrows()))
top_city, top_city_row = next(iter(city_summary.iterrows()))
top_payment, top_payment_row = next(iter(payment_summary.iterrows()))
top_product, top_product_row = next(iter(product_summary.iterrows()))

observations = [
    f'The dataset contains {overall_stats["Number of orders"]:,.0f} unique orders with total sales of {overall_stats["Total sales"]:,.2f}.',
    f'{top_category} is the highest-performing category, generating {top_category_row["total_sales"]:,.2f} in sales from {top_category_row["number_of_orders"]:,.0f} orders.',
    f'{top_city} is the leading city by sales ({top_city_row["total_sales"]:,.2f}).',
    f'{top_payment} is the most valuable payment method by revenue ({top_payment_row["total_sales"]:,.2f}).',
    f'The top product/category by revenue is {top_product}, with sales of {top_product_row["total_sales"]:,.2f}.'
]

print('Key observations')
for number, observation in enumerate(observations, start=1):
    print(f'{number}. {observation}')